# 10장. Plugins로 Skill과 MCP 묶어 배포하기

8장의 목표 MCP와 9장의 분석 Skill을 Plugin으로 묶습니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | Plugin ZIP·등록·두 기능의 사용 확인 |
| 이번 실습 조합 | 분석 Skill + 목표 MCP + 목표 비교 보고서 |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · Colab 업로드용


## 학습 안내

| 장 | 제공하는 구성 | 이번 장과의 연결 |
| --- | --- | --- |
| 8장 | MCP 도구 연결 | 목표 매출 조회 |
| 9장 | Skill ZIP | 분석 절차·참고 자료·실행 코드 |
| 10장 | Plugin ZIP | Skill과 MCP 설정을 하나의 배포 단위로 제공 |

이번 장은 `environment.plugins`만으로 두 기능을 제공합니다. `agent.tools`에 같은 MCP를 다시 등록하거나 `environment.skills`에 같은 Skill을 중복 첨부하지 않습니다. Plugin은 새로운 분석 알고리즘이 아니라 기능을 함께 전달하는 패키지입니다.

수동 실습은 위에서 아래로 실행합니다. 자동 검증은 `uv run --group dev python scripts/run_extensions.py --chapter 10 --live`로 실행합니다. Plugin과 Skill을 명시적으로 요청하므로 자동 선택 품질을 평가하지 않습니다.


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 환경 준비와 실행 모드

로컬에서는 `uv sync --locked` 후 `.venv` 커널을 선택합니다. Colab에서는 다운로드 안내의 독립 실행본을 업로드합니다. 준비 셀은 패키지를 직접 설치하고 포함 파일을 생성합니다.
아래 RUN_LIVE를 True로 바꿨을 때만 실제 API 호출·sandbox 생성이 발생합니다.
기존 `.env.local`의 키·모델을 재사용하며 키는 sandbox·ZIP·로그에 넣지 않습니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


In [ ]:
import json
from tutorial_helpers.common import load_settings

RUN_LIVE = False
MODEL_OVERRIDE = ""
CHAPTER = "10_plugins"
client = None
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
print({"live_enabled": settings.enabled, "model": settings.model or "Offline mode"})


## 1. Plugin 구성과 파일 연결 확인하기

```text
sales-reporting/
├── .codex-plugin/plugin.json
├── .mcp.json
├── skills/csv-report/
│   ├── SKILL.md
│   ├── references/report-schema.md
│   └── scripts/analyze_sales.py
├── servers/sales_targets_server.py
└── assets/sales_targets.json
```

| 파일/설정 | 역할 |
| --- | --- |
| `.codex-plugin/plugin.json` | Plugin 이름·설명·로컬 버전과 Skill/MCP 경로 선언 |
| `.mcp.json` | stdio 서버의 command·args·cwd 설정 |
| Skill 디렉터리 | CSV 분석 지침·계약·실행 코드 |
| 서버·목표 JSON | 합성 목표 매출을 반환하는 MCP와 데이터 |

다음 셀은 입력과 주요 파일을 읽어 표시합니다. 로컬에서 Skill이나 MCP 서버를 실행하지 않습니다. 서버의 stdout은 MCP 통신에 사용하고 진단 로그는 stderr로 분리합니다. Plugin 원본은 준비 과정에서 변경하지 않습니다.


In [ ]:
input_path = settings.root / "data/sales.csv"
csv_bytes = input_path.read_bytes()
csv_text = csv_bytes.decode("utf-8")
folder = settings.root / "plugins/sales-reporting"
manifest = json.loads((folder / ".codex-plugin/plugin.json").read_text(encoding="utf-8"))
mcp_config = json.loads((folder / ".mcp.json").read_text(encoding="utf-8"))
print(csv_text)
for name in (".codex-plugin/plugin.json", ".mcp.json", "skills/csv-report/SKILL.md",
             "skills/csv-report/references/report-schema.md",
             "skills/csv-report/scripts/analyze_sales.py", "servers/sales_targets_server.py",
             "assets/sales_targets.json"):
    print(f"\n--- {name} ---")
    print((folder / name).read_text(encoding="utf-8"))


### Plugin의 상대 경로와 설치 의존성

`skills`와 `mcpServers`는 manifest에서 연결할 파일/디렉터리를 가리킵니다. 이 Plugin의 `.mcp.json`에서 `cwd="."`는 Plugin 루트입니다. 따라서 `servers/sales_targets_server.py`와 `assets/sales_targets.json`도 그 루트에서 찾습니다. 설치된 Plugin의 절대 경로를 하드코딩하지 않습니다.

Plugin의 MCP 설정 형식은 `agent.tools`의 MCP 요청 스키마와 다릅니다. 8장의 필드를 그대로 복사하지 마세요. 서버 프로세스 시작은 원격 executor가 담당하며 `tool_handlers`나 수동 tool_result 제출은 사용하지 않습니다.

ZIP에는 코드·설정·합성 데이터만 포함합니다. MCP Python 패키지는 `environment.packages`로 따로 설치합니다. hosted stdio MCP의 네트워크 정책은 생략하거나 enabled여야 하므로 여기서는 enabled를 유지합니다. 이 설정은 패키지 다운로드만을 위한 설명이 아닙니다. [공식 Plugin 가이드](https://developers.openai.com/api/docs/guides/agents-api/tools/plugins#authenticate-mcp-servers)


## 2. ZIP과 inline Plugin 요청 준비하기

최상위 `sales-reporting/` 하나를 포함하는 ZIP을 만듭니다. 요청 name·description은 manifest에서 읽어 일치시킵니다. 목록·크기·SHA-256은 로컬 번들 식별 정보이며 원격 실행 증거가 아닙니다. manifest version도 로컬 선언이며 등록 응답이 버전이나 코드 해시를 반환한다고 가정하지 않습니다.

`archive_folder()`는 파일 포장만 합니다. 가상환경·설치된 패키지·API 키를 넣지 않습니다. Skill·서버·데이터 복사본을 바꿀 때는 노트북 안에서 자동 동기화하지 말고 `scripts/sync_plugin_sources.py`를 명시적으로 실행해 diff를 검토합니다. `scripts/build_bundles.py`는 원본 일치 여부를 검사하고 `_build/bundles/`를 생성합니다.


In [ ]:
from tutorial_helpers.extensions import archive_folder
import base64
import hashlib
import io
import zipfile

raw = archive_folder(folder)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    print("ZIP entries:", archive.namelist())
print({"name": manifest["name"], "version": manifest["version"],
       "description": manifest["description"], "zip_bytes": len(raw),
       "zip_sha256": hashlib.sha256(raw).hexdigest()})


In [ ]:
MCP_VERSION = "1.26.0"
plugin = {
    "type": "inline",
    "name": manifest["name"],
    "description": manifest["description"],
    "source": {
        "type": "base64",
        "media_type": "application/zip",
        "data": base64.b64encode(raw).decode("ascii"),
    },
}
environment = {
    "type": "openai_hosted",
    "container_size": "small",
    "network": {"access": "enabled"},
    "packages": {"python": [f"mcp=={MCP_VERSION}"]},
    "plugins": [plugin],
    "files": [{"type": "inline", "path": "/workspace/sales.csv",
               "data": base64.b64encode(csv_bytes).decode("ascii")}],
}
print({"plugin_name": plugin["name"], "plugin_type": plugin["type"],
       "source_type": plugin["source"]["type"], "packages": environment["packages"],
       "sandbox_input": environment["files"][0]["path"]})


### 분석 요청과 결과 계약

요청문은 실행 셀 밖에서 준비합니다. Skill 번들 스크립트가 기본 지표와 세 파일을 만들고, MCP가 합성 목표를 조회하며, 에이전트가 보고서에 목표 비교를 덧붙입니다. 두 기능의 역할을 섞지 않습니다.

| 결과 | 형식 |
| --- | --- |
| `summary.json` | 기본 metrics만 유지 |
| 최종 응답 | metrics·target_revenue·attainment_rate의 확장 JSON |
| `report.md` | 기본 보고서 + 목표 매출·달성률(백분율) |
| `skill_evidence.json` | Skill 이름·revision 표식·입력 바이트 해시 |

JSON의 attainment_rate는 0~1 등의 비율이고 보고서는 100을 곱한 백분율입니다. 목표를 초과하면 비율은 1보다 클 수 있습니다. 보고서의 추가 두 줄은 검증 가능한 명시적 형식으로 요청합니다.


In [ ]:
BASE_INSTRUCTIONS = (
    "You are a careful sales analyst. Use Python "
    "for all calculations. Use only the provided "
    "CSV; never invent rows. Do not browse the web"
    " or delegate. Keep progress commentary short."
    " Return only the JSON object requested by the"
    " user as the final answer, with no Markdown "
    "fences."
)
METRICS_CONTRACT = (
    'Base metrics must be exactly {"row_count": integer, '
    '"total_quantity": integer, "total_revenue": '
    'integer, "by_product": {product_name: '
    "integer_revenue}}. Revenue is quantity "
    "multiplied by unit_price. Do not round "
    "integer amounts."
)

ENRICHED_CONTRACT = (
    'Return only JSON with "metrics", "target_revenue" (integer), and '
    '"attainment_rate" = total_revenue / target_revenue as a unit fraction, not a percentage. '
    'metrics must equal parsed summary.json. Keep summary.json in the base metrics format.'
)
task = (
    "Use the csv-report skill from the sales-reporting plugin. Read its SKILL.md and locate "
    "the bundled scripts/analyze_sales.py relative to the skill directory. Run it with Python "
    "using --input /workspace/sales.csv --output-dir /workspace/outputs; do not rewrite it. "
    "Call the plugin's get_sales_target MCP tool for 2026-01 exactly once. "
    "Read summary.json, report.md, and skill_evidence.json. Append two lines to report.md: "
    "'Target revenue: <integer target>' and 'Attainment: <percentage with two decimal places>%'. "
    + METRICS_CONTRACT + " " + ENRICHED_CONTRACT
)


## 3. Plugin을 첨부해 분석 실행하기

준비한 environment와 task를 `sessions.create()`에 전달합니다. 등록·패키지 설치·MCP 시작·Skill 실행은 원격 단계이며 앞의 셀은 로컬에서 파일을 읽고 포장했을 뿐입니다. Plugin을 수정했으면 새 실행에서 새 세션을 만듭니다. 기존 세션의 자동 reload는 가정하지 않습니다.

각 실행·확인 단계는 `cleanup_on_error()`로 오류·일반 인터럽트에서 정리를 시도합니다. 정상 경로에서는 **8절 정리까지 실행**하세요. 단계 사이에 학습을 멈출 때도 정리하며, 강제 커널 종료는 자동 정리를 보장하지 않습니다.


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError(
        "Set RUN_LIVE=True to run the API example. Use the validation script for offline checks."
    )
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)


### 이번 실행 상태 준비

복구 기록은 `outputs/extensions/<chapter>/<run_id>/`, 다운로드는 그 아래 `downloads/<turn_id>/`에 저장합니다. 새 실행 시작 시 상태 준비를 한 번만 실행하세요. 진행 중 재실행하면 자원 식별자를 잃을 수 있습니다.

타이머는 실행 직전 시작해 셀 사이 학습 시간을 제외합니다. 600초 제한은 이벤트 도착 시 확인하므로 무응답 중 강제 중단하는 전체 타이머가 아닙니다. HTTP timeout은 90초입니다.


In [ ]:
from tutorial_helpers.common import (
    new_session_metadata, cleanup_resources, cleanup_on_error, check_event, safe_error,
)
from uuid import uuid4
import time


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/extensions" / CHAPTER / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
result = turn = None
downloads = None
downloads_complete = False
installed_environment = None


### 세션 생성과 응답 스트리밍

텍스트 delta를 표시하며 최종 응답을 수집하고, 스트림을 닫은 뒤 root Turn을 조회합니다. 새 작업의 중복 제출은 차단합니다. 실패·취소를 감지하고 예상하지 않은 외부 action은 오류로 처리합니다.

Turn 완료·모델의 Skill 언급·JSON 파싱만으로 Skill 사용이 확인됐다고 판단하지 않습니다. 다음 두 절에서 등록과 실제 명령을 따로 확인합니다. 이벤트 수·순서를 고정하거나 `idle`만으로 성공을 판단하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if attempts:
        raise RuntimeError("This run already attempted session creation. Do not resubmit.")
    started = time.monotonic()
    with client.beta.agents.sessions.create(
        agent={"model": settings.model, "instructions": BASE_INSTRUCTIONS},
        environment=environment,
        input=task,
        metadata=new_session_metadata(CHAPTER, run_id, attempts=attempts, output_dir=output_dir),
        stream=True,
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.requires_action":
                raise RuntimeError("Unexpected external action in this hosted exercise")
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        result = stream.get_final_result()
        sessions.setdefault(result.session_id, {"deleted": False})

    turn = client.beta.agents.sessions.turns.retrieve(
        result.turn_id, session_id=result.session_id, timeout=15
    )
    if result.turn.status != "completed" or turn.status != "completed":
        raise RuntimeError("Root turn completion was not confirmed")
    print("\nCompleted analysis:")
    print({"session_id": result.session_id, "turn_id": result.turn_id, "status": turn.status})
    print(json.dumps(json.loads(result.output_text), indent=2))


## 4. 이번 환경의 Plugin 등록 확인하기

Session의 Environment ID로 `environments.retrieve()`를 호출합니다. 이벤트에서 ID를 놓쳤다면 Session 재조회로 복구합니다. 환경 ID와 Plugin 이름/inline 유형을 확인하고 이름·유형·설명만 출력합니다.

이 응답은 등록 metadata입니다. ZIP 내부 내용·manifest 버전·실제 MCP 호출·Skill 실행을 증명하지 않습니다. Plugin 내부 Skill이 별도 skills 목록에도 반드시 나타난다고 가정하지 않습니다. 다음 절에서 실제 동작을 확인합니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before inspecting registration.")
    environment_id = sessions[result.session_id].get("environment_id")
    if not environment_id:
        session = client.beta.agents.sessions.retrieve(result.session_id, timeout=15)
        environment_id = getattr(session.environment, "id", None)
        sessions[result.session_id]["environment_id"] = environment_id
    if not environment_id:
        raise RuntimeError("No environment ID was captured for this session")
    installed_environment = client.beta.agents.environments.retrieve(environment_id, timeout=15)
    if installed_environment.id != environment_id:
        raise RuntimeError("Retrieved environment differs from the session environment")
    print("Registered Plugins:", [
        {"name": s.name, "type": s.type, "description": s.description}
        for s in installed_environment.plugins
    ])
    if not any(s.name == plugin["name"] and s.type == "inline" for s in installed_environment.plugins):
        raise RuntimeError("The requested inline Plugin is not registered")


## 5. MCP 호출과 Skill 실행을 각각 확인하기

`items.list()`를 SDK 페이지 반복으로 읽고 이번 root Turn의 MCP 호출과 명령을 따로 표시합니다. MCP 인자·상태·오류·반환값을 보고, Python 명령은 스크립트 경로·입력/출력 인자·cwd·상태·종료 코드를 확인하세요. None은 성공 종료 코드 0이 아닙니다. 파일 읽기와 스크립트 실행도 구분합니다.

MCP 설정 키는 sales_targets이고 서버 코드의 FastMCP 이름은 sales-targets입니다. 런타임 server_label은 관측 값으로 표시하며 Plugin 네임스페이스 형식을 추측해 고정하지 않습니다. metadata만으로 호출 출처를 독립 증명하지 않습니다. 검증기는 root Turn의 정확한 도구 이름·기간·1회 호출·완료·오류 없음, 명시적 Python CLI와 인자를 검사합니다. 계산값이 맞거나 Turn이 완료됐어도 실패 호출은 통과하지 않습니다.

기본 확인에는 Items API를 사용합니다. 복잡한 실행을 조사할 때만 Trace를 추가로 활용합니다. 본문은 원격 MCP stderr 수집이나 일반 shell 프로그램 분석을 구현하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before inspecting execution history.")
    items = list(client.beta.agents.sessions.items.list(
        result.session_id, order="asc", limit=100, timeout=15,
    ))
    mcp_calls = [i for i in items if i.type == "mcp_call" and i.turn_id == result.turn_id]
    command_items = [i for i in items if i.type == "command_execution" and i.turn_id == result.turn_id]
    print("Plugin MCP configuration keys:", list(mcp_config["mcpServers"]))
    print("MCP calls from this turn:")
    for item in mcp_calls:
        print({"item_id": item.id, "server_label": item.server_label, "name": item.name,
               "turn_id": item.turn_id, "arguments": item.arguments, "status": item.status,
               "error": item.error, "output": item.output})
    print("Commands from this turn:")
    for item in command_items:
        print({"item_id": item.id, "turn_id": item.turn_id, "command": item.command,
               "cwd": item.cwd, "status": item.status, "exit_code": item.exit_code,
               "output": item.output})


## 6. 세 결과 파일 다운로드하기

`artifacts.for_result(result)`로 완료한 Session·Turn에 범위를 고정합니다. 정확한 경로를 다운로드하고 원격 artifact와 로컬 경로·크기를 표시합니다. SDK가 누락·중복 경로를 오류로 처리합니다.

세 파일을 모두 받은 뒤에만 완료로 표시합니다. 재실행은 세션 정리 전에 한정합니다. 오류 시 정리를 시도하므로 같은 세션에서 재시도를 보장하지 않으며 부분 파일을 성공으로 처리하거나 분석을 자동 재실행하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before downloading artifacts.")
    downloads_complete = False
    downloads = output_dir / "downloads" / result.turn_id
    downloads.mkdir(parents=True, exist_ok=True)
    scoped = client.beta.agents.sessions.artifacts.for_result(result)
    for name in ("summary.json", "report.md", "skill_evidence.json"):
        local_path = downloads / name
        artifact = scoped.download(f"/workspace/outputs/{name}", to=local_path, timeout=30)
        print({"artifact_id": artifact.id, "turn_id": artifact.turn_id,
               "sandbox_path": artifact.path, "local_path": str(local_path),
               "local_bytes": local_path.stat().st_size})
    downloads_complete = True


## 7. 기본 지표와 목표 비교 결과 읽기

최종 확장 JSON과 기본 summary.json을 나란히 읽습니다. metrics는 기본 summary와 같아야 하고, 목표는 MCP 응답에서 가져오며 JSON 달성률은 비율입니다. 보고서에서는 목표 매출과 두 자리 소수 백분율의 추가 줄을 확인하세요. 다음 셀은 정상 경로에서 추가 API 호출 없이 로컬 결과만 표시합니다.

skill_evidence의 입력 SHA-256은 실제 전송한 CSV 바이트 식별 정보이고, script_revision은 코드에 정의한 표식입니다. ZIP 해시·revision 문자열·등록 정보만으로 변조 방지나 독립적인 실행 증명을 보장하지 않습니다. 정답·보고서 추가 줄·해시 대조는 삭제 전 검증 사본에서만 수행합니다.

목표 JSON도 같은 환경 안에 있으므로 이 실습은 목표 데이터의 접근 격리를 증명하지 않습니다. MCP 실제 호출 여부와 계산 결과를 구분해 읽으세요.


In [ ]:
from IPython.display import Markdown, display

with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if not downloads_complete:
        raise RuntimeError("Complete all three downloads before reading local results.")
    actual = json.loads(result.output_text)
    downloaded_summary = json.loads((downloads / "summary.json").read_text(encoding="utf-8"))
    skill_evidence = json.loads((downloads / "skill_evidence.json").read_text(encoding="utf-8"))
    report_text = (downloads / "report.md").read_text(encoding="utf-8")
    print("Final response:")
    print(json.dumps(actual, indent=2))
    print("Downloaded summary.json:")
    print(json.dumps(downloaded_summary, indent=2))
    print("Downloaded skill_evidence.json:")
    print(json.dumps(skill_evidence, indent=2))
    display(Markdown(report_text))


## 8. 세션 정리와 로컬 결과 보관

필요한 세 파일을 먼저 내려받고 정리 셀을 실행합니다. helper는 고유 metadata로 이번 실행의 소유 세션을 찾아 필요하면 취소하고 삭제·재조회 404로 API상 부재를 확인합니다. deleted·verified_absent를 확인하세요. 물리적인 MCP 프로세스 종료나 sandbox 정리의 동기 완료를 독립 확인하는 검사는 아닙니다.

오류 정리와 마지막 finally는 원래 실행/검증 오류를 보존합니다. 검증 도구는 삭제 전에 이력·파일 검사를 추가하고 검사 실패에도 정리를 시도합니다. 정상 정리 후 client는 닫히므로 기존 세션으로 API 확인을 계속하지 않습니다. 로컬 Plugin 원본·ZIP 바이트·다운로드·결과 객체는 남습니다.

이 예제는 별도 영구 Plugin 설치 리소스나 저장 Agent를 만들지 않습니다. 세션에 제공한 inline Plugin과 Codex Desktop 사용자의 Plugin 설치/감시는 다른 범위입니다. 수동 Jupyter 실행에는 receipt가 없고 실행 사본은 outputs/extensions-validation/, 결과/정리 기록은 outputs/extensions/에 있습니다.


In [ ]:
try:
    print("Cleaning up this run's session...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
print({"session_cleanup": sessions})


## 9. 실패 시 조치와 다음 장

| 단계 | 확인할 내용 |
| --- | --- |
| 번들 | manifest 경로·단일 최상위 폴더·원본 복사본 일치 |
| 환경·등록 | 패키지 설치·network 설정·Environment ID·Plugin 이름/유형 |
| MCP | 서버 cwd/args·도구 인자·호출 상태/오류 |
| Skill | 번들 스크립트 경로·CLI 인자·종료 코드·입력 해시 |
| 결과 | 세 파일·최종 JSON과 summary 구분·목표/백분율 추가·정리 |

실패하거나 부분 파일만 내려받았다면 성공으로 판단하지 말고 정리 후 원인을 확인합니다. 자동 재제출하지 않습니다. 오류 시 정리로 client/세션이 닫힐 수 있으므로 같은 실행의 재시도를 보장하지 않습니다. 키·원본 API 오류 body는 출력하지 않습니다. 강제 종료 복구에는 `creation.json`의 metadata와 Session ID를 사용하며 [빠른 시작](../quickstart.ipynb)을 참고하세요.

등록과 두 기능의 실제 사용, 비율과 백분율, API상 정리와 로컬 보관을 설명할 수 있으면 완료입니다. [11장 승인](11_guardrails_approvals.ipynb)에서는 보고서를 게시하기 전 사용자 판단과 호출 결과 캐시를 추가합니다. 현재 10장은 목표 조회/보고서 작성만 하며 외부 게시를 하지 않습니다. self-hosted 등록·환경 template 재사용은 공식 문서의 선택 확장으로 두고 이번 실습에 섞지 않습니다.


## 공식 참고 자료

### Agents API 전용 문서

- [Plugins](https://developers.openai.com/api/docs/guides/agents-api/tools/plugins) — manifest·Skill/MCP 패키징·hosted ZIP과 self-hosted 등록.
- [MCP connections](https://developers.openai.com/api/docs/guides/agents-api/tools/mcp) — MCP 연결 위치·stdio/HTTP·인증·도구 접근 제어.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
